# Session 06 practical lab: Regression challenge for expected market price

> Warning: This notebook is a teaching lab. The production baseline notebook remains `04_regression_audi_a3_germany.ipynb`.

This lab runs directly from processed CSV files. It does not require cloud credentials or warehouse access.

Head of Data 101 uses one central idea: **the pipeline is the product**.

In this lab, regression creates an expected market price reference layer for used-vehicle acquisition analysis. The model output is `expected_price_eur`. The business signal is `expected_price_gap = actual price - expected price`.

A negative gap means the listing is cheaper than the model expects. A positive gap means the listing is more expensive than the model expects. The gap is not an automatic investment recommendation; it is a prioritization signal for analyst review.

Course narrative boundary:

- Regression predicts `expected_price_eur`.
- Regression produces `expected_price_gap`.
- Classification remains separate and predicts `top_price`.
- BI later combines actual price, expected-price gap, and top-price probability.


## 3-hour live session agenda

- 00:00-00:15 - Data contract and business objective
- 00:15-00:35 - Visual market reading
- 00:35-01:05 - Simple regression baseline
- 01:05-01:45 - Multivariate expected-price model
- 01:45-02:15 - Model diagnostics and interpretation
- 02:15-02:45 - From expected price to review queue
- 02:45-03:00 - Discussion and BI hand-off


## Minimum viable live path

If class time becomes tight, complete Challenge 0, Challenge 1, Challenge 2, Challenge 3 and Challenge 4; leave the optional flexible models for after class.


## How to work

This is not a pure coding course. Use AI-assisted coding when useful, but make sure you can explain every result.

For each challenge, read the business reason, complete the small coding task or inspect the instructor-led output, and write a short business interpretation in English.

Look for the labels **Student task**, **Instructor-led demo**, **Discussion pause**, and **Optional extension**. They are there to protect class pacing.


## Challenge 0 - Data contract and modeling dataset

**Suggested time:** 15 minutes

**Instructor-led demo**

**Business reason:**  
Before modeling, a professional data scientist must know exactly which dataset contract is being consumed. This is not a path-handling exercise; it is a contract-awareness exercise.

**Objective:**  
Load a processed CSV, validate the required regression columns, create `listing_id` if needed, and prepare a clean modeling dataframe called `df`.

**Required columns:** `price_eur`, `mileage_km`, `age_years`, `power_hp`.

**Expected output format:**
- Selected CSV path.
- Row count before and after filtering.
- Available columns.
- First 5 rows.
- Missing values summary.

**Discussion pause:**
- Which CSV did the notebook load?
- Why should required columns fail loudly instead of being guessed?


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.2f}".format)
plt.style.use("default")


def find_repo_root(start):
    for path in [start] + list(start.parents):
        if (path / ".git").exists() or (path / "config" / "project_config.yaml").exists():
            return path
    return start


def load_project_config(config_path):
    config = {}
    if not config_path.exists():
        return config
    for raw_line in config_path.read_text(encoding="utf-8-sig").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or ":" not in line:
            continue
        key, value = line.split(":", 1)
        key = key.strip()
        value = value.strip()
        if value.startswith(("'", '"')) and value.endswith(("'", '"')):
            value = value[1:-1]
        elif value.lower() in ("true", "false"):
            value = value.lower() == "true"
        else:
            try:
                value = int(value)
            except ValueError:
                try:
                    value = float(value)
                except ValueError:
                    pass
        config[key] = value
    return config


def relative_path(path):
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)


def csv_missing_required_columns(path, required_columns):
    try:
        columns = pd.read_csv(path, nrows=0).columns
    except Exception as exc:
        return required_columns, str(exc)
    missing = [column for column in required_columns if column not in columns]
    return missing, None


def csv_contains_required_columns(path, required_columns):
    missing, error = csv_missing_required_columns(path, required_columns)
    return not missing and error is None


def newest_csv_in_folder(folder):
    if not folder.exists():
        return None
    csv_files = [path for path in folder.glob("*.csv") if path.is_file()]
    if not csv_files:
        return None
    return max(csv_files, key=lambda path: path.stat().st_mtime)


def newest_data_csv_with_required_columns(data_root, required_columns):
    if not data_root.exists():
        return None
    candidates = []
    for path in data_root.rglob("*.csv"):
        if path.is_file() and csv_contains_required_columns(path, required_columns):
            candidates.append(path)
    if not candidates:
        return None
    return max(candidates, key=lambda path: path.stat().st_mtime)


REQUIRED_COLUMNS = ["price_eur", "mileage_km", "age_years", "power_hp"]
SESSION_06_FULL_CSV = "autoscout24_listings_processed_audi_a3_germany_20251228_205210.csv"
OPTIONAL_COLUMNS = [
    "listing_id",
    "make",
    "model",
    "brand",
    "fuel_type",
    "listing_country",
    "registration_year",
    "registration_month",
    "price_label",
    "price_outlier_iqr",
    "mileage_outlier_iqr",
    "power_outlier_iqr",
    "logical_issue",
]


def select_processed_csv(required_columns, preferred_filename):
    preferred_paths = [
        ("Preferred full classroom sample CSV (local)", processed_folder / preferred_filename),
        ("Repo-visible fallback sample", sample_processed_folder / preferred_filename),
    ]
    rejected = []

    for source_label, candidate in preferred_paths:
        if candidate.exists():
            missing, error = csv_missing_required_columns(candidate, required_columns)
            if not missing and error is None:
                return candidate, source_label
            rejected.append((source_label, candidate, missing, error))

    candidate = newest_csv_in_folder(processed_folder)
    if candidate is not None:
        missing, error = csv_missing_required_columns(candidate, required_columns)
        if not missing and error is None:
            return candidate, "Latest local processed CSV"
        rejected.append(("Latest local processed CSV", candidate, missing, error))

    candidate = newest_data_csv_with_required_columns(PROJECT_ROOT / "data", required_columns)
    if candidate is not None:
        return candidate, "Last-resort scan for any CSV with required columns"

    details = []
    for source_label, path, missing, error in rejected:
        if error:
            details.append(f"{source_label}: {relative_path(path)} could not be read ({error})")
        else:
            details.append(f"{source_label}: {relative_path(path)} missing {missing}")
    raise FileNotFoundError(
        "No processed CSV with the required regression columns was found. "
        "Run Notebook 02 to create data/processed/*.csv, or use data/sample/processed/.\n"
        + "\n".join(details)
    )


def evaluate_regression_model(model_name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return {
        "model_name": model_name,
        "mae_test": mean_absolute_error(y_test, y_pred),
        "rmse_test": float(np.sqrt(mean_squared_error(y_test, y_pred))),
        "r2_test": r2_score(y_test, y_pred),
        "predictions": y_pred,
    }


def plot_actual_vs_predicted(y_true, y_pred, title):
    plt.figure(figsize=(6, 5))
    plt.scatter(y_true, y_pred, alpha=0.35)
    min_value = min(float(np.min(y_true)), float(np.min(y_pred)))
    max_value = max(float(np.max(y_true)), float(np.max(y_pred)))
    plt.plot([min_value, max_value], [min_value, max_value], color="black", linewidth=1)
    plt.title(title)
    plt.xlabel("Actual price (EUR)")
    plt.ylabel("Predicted price (EUR)")
    plt.tight_layout()
    plt.show()


def plot_residuals(y_true, y_pred, title):
    residuals = y_true - y_pred
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].hist(residuals, bins=30, color="#2563eb", alpha=0.8)
    axes[0].set_title("Residual distribution")
    axes[0].set_xlabel("Actual - predicted (EUR)")
    axes[0].set_ylabel("Listings")
    axes[1].scatter(y_pred, residuals, alpha=0.35)
    axes[1].axhline(0, color="black", linewidth=1)
    axes[1].set_title("Residuals vs predicted price")
    axes[1].set_xlabel("Predicted price (EUR)")
    axes[1].set_ylabel("Residual (EUR)")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def plot_price_relationship(data, feature, xlabel):
    plt.figure(figsize=(7, 4.5))
    plt.scatter(data[feature], data["price_eur"], alpha=0.35)
    plt.title(f"Price vs {xlabel}")
    plt.xlabel(xlabel)
    plt.ylabel("Price (EUR)")
    plt.tight_layout()
    plt.show()


PROJECT_ROOT = find_repo_root(Path.cwd())
PROJECT_CONFIG = load_project_config(PROJECT_ROOT / "config" / "project_config.yaml")
RANDOM_SEED = int(PROJECT_CONFIG.get("random_state", 42))
np.random.seed(RANDOM_SEED)

make_scope = str(PROJECT_CONFIG.get("make", "Audi")).strip()
model_scope = str(PROJECT_CONFIG.get("model", "A3")).strip()
country_scope = str(PROJECT_CONFIG.get("country", "Germany")).strip()
TAG = f"{make_scope}_{model_scope}_{country_scope}".lower().replace(" ", "_")

processed_folder = PROJECT_ROOT / str(PROJECT_CONFIG.get("processed_data_path", "data/processed"))
sample_processed_folder = PROJECT_ROOT / "data" / "sample" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

selected_csv_path, selected_source = select_processed_csv(REQUIRED_COLUMNS, SESSION_06_FULL_CSV)
df_raw = pd.read_csv(selected_csv_path)
row_count_before = len(df_raw)
missing_required = [column for column in REQUIRED_COLUMNS if column not in df_raw.columns]
if missing_required:
    raise ValueError("Missing required regression columns: " + ", ".join(missing_required))

missing_optional = [column for column in OPTIONAL_COLUMNS if column not in df_raw.columns]
df = df_raw.copy()
for column in REQUIRED_COLUMNS:
    df[column] = pd.to_numeric(df[column], errors="coerce")

valid_modeling_rows = (
    df["price_eur"].gt(0)
    & df["mileage_km"].gt(0)
    & df["age_years"].ge(0)
    & df["power_hp"].gt(0)
)
rows_removed = int((~valid_modeling_rows).sum())
df = df.loc[valid_modeling_rows].reset_index(drop=True)
if "listing_id" not in df.columns:
    df.insert(0, "listing_id", np.arange(1, len(df) + 1))
    missing_optional = [column for column in missing_optional if column != "listing_id"]

if df.empty:
    raise ValueError("The selected CSV has no valid modeling rows after filtering.")

print("Selected CSV:", relative_path(selected_csv_path))
print("Source priority:", selected_source)
print("Configured scope:", make_scope, model_scope, country_scope)
print("Rows before filtering:", row_count_before)
print("Rows after filtering:", len(df))
print("Rows removed by required-value filter:", rows_removed)
print("Available columns:")
print(", ".join(df.columns))
if missing_optional:
    print("Optional columns not found in this CSV:", ", ".join(missing_optional))

print("\nFirst 5 rows:")
display(df.head())

print("Missing values summary:")
missing_summary = df.isna().sum().sort_values(ascending=False).to_frame("missing_values")
display(missing_summary[missing_summary["missing_values"] > 0])


## Challenge 1 - Visual market reading before modeling

**Suggested time:** 20 minutes

**Student task**

**Business reason:**  
A data scientist should read the market visually before fitting a model. Charts reveal relationships, outliers, non-linearity, and possible data issues that metrics alone can hide.

**Objective:**  
Inspect the relationship between price and the main explanatory variables.

**Expected output format:**
- Three scatter plots.
- Three short visual insights.
- One hypothesis about the strongest predictor.


In [ ]:
# Instructor-led plotting helper; student task is interpretation.
plot_df = df.sample(n=min(20000, len(df)), random_state=RANDOM_SEED)

plot_price_relationship(plot_df, "mileage_km", "mileage (km)")
plot_price_relationship(plot_df, "age_years", "age (years)")
plot_price_relationship(plot_df, "power_hp", "power (hp)")


### Your visual insights

1. Mileage insight:
2. Age insight:
3. Power insight:
4. Hypothesis about the strongest predictor:


## Challenge 2 - One simple baseline model

**Suggested time:** 30 minutes

**Student task**

**Business reason:**  
Before building a serious expected-price model, we need a simple baseline. A more complex model must beat a simple and explainable rule.

**Objective:**  
Train one baseline model: `price_eur ~ age_years`.

**Expected output format:**
- Baseline metrics: MAE, RMSE, R2.
- Coefficient and intercept.
- One plain-English coefficient interpretation.

**Discussion pause:**
- What does MAE mean in euros?
- Does the coefficient sign make business sense?
- Is one variable enough for decision support?


In [ ]:
# Challenge 2 scaffold.
# Train one simple LinearRegression model: price_eur ~ age_years.

# Your code here
# train_df, test_df = train_test_split(df, test_size=0.20, random_state=RANDOM_SEED)
# baseline_features = ["age_years"]
# X_train = train_df[baseline_features]
# X_test = test_df[baseline_features]
# y_train = train_df["price_eur"]
# y_test = test_df["price_eur"]
#
# baseline_model = LinearRegression()
# baseline_result = evaluate_regression_model(
#     "Age-only linear baseline",
#     baseline_model,
#     X_train,
#     X_test,
#     y_train,
#     y_test,
# )
#
# Build a one-row metric table and interpret the coefficient.


### Your baseline conclusion

1. Baseline variable used:
2. MAE in business language:
3. Coefficient interpretation:
4. Is this enough for decision support?


## Challenge 3 - Multivariate expected-price model

**Suggested time:** 40 minutes

**Student task with helper functions**

**Business reason:**  
A useful expected-price layer should control for several vehicle characteristics at the same time. Mileage, age, and power each explain part of the market.

**Objective:**  
Train `price_eur ~ mileage_km + age_years + power_hp` and compare it with the simple baseline.

**Expected output format:**
- Metric table comparing baseline and multivariate model.
- Coefficient table.
- Actual vs predicted plot.
- Residual plot.

**Discussion pause:**
- Did the model improve the euro error?
- Do the coefficient signs make economic sense?
- Where do residuals suggest the model is weak?


In [ ]:
# Challenge 3 scaffold.
# Use the helper functions to avoid repeating metric and plotting mechanics.

# Your code here
# core_features = ["mileage_km", "age_years", "power_hp"]
# multivariate_model = LinearRegression()
# multivariate_result = evaluate_regression_model(
#     "Core multivariate linear regression",
#     multivariate_model,
#     train_df[core_features],
#     test_df[core_features],
#     y_train,
#     y_test,
# )
#
# Compare baseline_metrics_df with the multivariate metrics.
# Build a coefficient table from multivariate_model.coef_.
# Use plot_actual_vs_predicted(...) and plot_residuals(...).


### Your multivariate conclusion

1. Did the multivariate model improve MAE?
2. Which coefficient is easiest to explain?
3. Where does the model seem weak?
4. Is it useful enough as an expected-price reference layer?


## Challenge 4 - From regression output to decision queue

**Suggested time:** 35 minutes

**Student task**

**Business reason:**  
A regression model becomes useful when it produces a decision-support artifact. The expected-price layer should help analysts prioritize which listings deserve review.

**Objective:**  
Train the selected model on all valid rows, generate expected-price outputs, and create a review queue.

**Expected output format:**
- `expected_price_eur`
- `expected_price_gap`
- `expected_price_gap_pct`
- Top cheaper-than-expected candidates.
- Top more-expensive-than-expected candidates.
- Review queue saved to `reports/{TAG}_regression_review_queue.csv`.

**Discussion pause:**
- Which 3 listings deserve analyst review first?
- Which listings may be model artifacts?
- What should BI show, and what should it avoid implying?


In [ ]:
# Challenge 4 scaffold.
# The selected model can be the core multivariate model unless you have a clear reason to choose otherwise.

# Your code here
# selected_model_name = "Core multivariate linear regression"
# selected_features = ["mileage_km", "age_years", "power_hp"]
# selected_model = LinearRegression()
# selected_model.fit(df[selected_features], df["price_eur"])
# expected_price = selected_model.predict(df[selected_features])
#
# review_queue = df[["listing_id", "price_eur", "mileage_km", "age_years", "power_hp"]].copy()
# review_queue["expected_price_eur"] = expected_price
# review_queue["expected_price_gap"] = review_queue["price_eur"] - review_queue["expected_price_eur"]
# review_queue["expected_price_gap_pct"] = review_queue["expected_price_gap"] / review_queue["expected_price_eur"]
#
# Sort to inspect cheaper-than-expected and more-expensive-than-expected listings.
# Save to REPORTS_DIR / f"{TAG}_regression_review_queue.csv".


### Your review queue conclusion

1. Three listings for analyst review:
2. One model limitation:
3. One next improvement:
4. What should BI show?


## Optional extension - Flexible models

**Optional extension**

Use this only if the class moves faster than expected or as after-class practice.

**Instructor-led demo**

Flexible models can capture non-linear relationships, but they may be harder to explain. Compare the core linear model with:

- `DecisionTreeRegressor(max_depth=4, random_state=RANDOM_SEED)`
- `RandomForestRegressor(n_estimators=100, max_depth=8, random_state=RANDOM_SEED, n_jobs=-1)`
- `HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_leaf_nodes=31, random_state=RANDOM_SEED)`

Discussion:
- Is the improvement large enough to matter in euros?
- Which model would be easiest to defend to a committee?
- What could go wrong if we only chase lower error?


In [ ]:
# Optional extension scaffold.
# Run this after the core lab only if there is time.

# Your code here
# advanced_models = {
#     "Decision Tree": DecisionTreeRegressor(max_depth=4, random_state=RANDOM_SEED),
#     "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=8, random_state=RANDOM_SEED, n_jobs=-1),
#     "Histogram Gradient Boosting": HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_leaf_nodes=31, random_state=RANDOM_SEED),
# }
# Compare these models with the core multivariate model using evaluate_regression_model(...).


## Final discussion: what did we learn about regression as a decision-support layer?

Use these prompts for the final class discussion:

- Why did we inspect the market visually before modeling?
- Why did we build a simple baseline?
- What is the business meaning of MAE, RMSE, and R2?
- How do residuals become possible mispricing signals?
- Why is `expected_price_gap` a prioritization signal rather than a decision?
- How should BI combine actual price and expected-price gap?
- Why does model output still require analyst judgment?

Final message:

**A useful regression model does not end in a metric. It ends in a reference layer that helps people explain deviations, prioritize review, and make better decisions.**
